# 语义搜索与RAG

语义搜索(semantic search)，其核心在于通过语义理解而非简单的关键词匹配来实现精准检索。
RAG（Retrieval-Augmented Generation）则是一种基于语义搜索的生成式模型，它将语义搜索的结果与生成式模型的输出相结合，以生成更符合用户需求的内容。

与此同时，文本生成模型的快速普及使得用户开始期待其提供事实性回答。尽管模型能够流畅自信地输出答案，但其内容在准确性和时效性方面仍存在不足。这种现象被称为“幻觉”​，而解决该问题的主要方法之一，便是构建能够实时检索相关信息并输入LLM的系统，从而生成有事实依据的答案。这种被称为RAG（检索增强生成）的技术，现已成为LLM最受瞩目的应用场景。

## 8.1 语义搜索与RAG技术全景

研究成果。当前主流技术可分为三大类：稠密检索(dense retrieval)、重排序(reranking)和RAG。

稠密检索：
  该技术基于文本嵌入（与前述章节原理相同）​，将搜索问题转化为查询向量与文档向量的最近邻匹配过程。图8-1直观展示了稠密检索的工作流程：接收搜索请求后，系统在文档库中进行向量比对，最终输出相关性最高的结果集合。

<div align="center">
    <img src="./picture/8-1.jpg">
    <p>图8-1：稠密检索是语义搜索的第一大核心类型，通过文本嵌入的相似度实现精准的结果筛选
</p>
</div>


重排序：搜索系统多采用多阶段处理流程。重排序模型作为其中关键环节，负责对初步检索结果进行相关性评分，并据此优化排序。图8-2展示了重排器与稠密检索的核心差异：前者需要接收来自前序搜索流程的中间结果作为输入基准。

<div align="center">
    <img src="./picture/8-2.jpg">
    <p>图8-2：重排器是语义搜索的第二大核心类型，重排器能够接收搜索查询与初始结果集，并根据相关性进行重新排序，从而显著提升结果质量</p>
</div>


RAG：随着文本生成模型能力的持续增强，一种融合查询响应功能的新型搜索系统应运而生——RAG，成为语义搜索的第三大类型。图8-3直观展现了此类生成式搜索系统的典型架构。

生成式搜索属于广义的RAG系统范畴。这类系统通过整合检索机制来增强文本生成功能，可有效抑制幻觉现象、提升事实准确性，并使模型输出与特定数据集保持逻辑一致性。

<div align="center">
    <img src="./picture/8-3.jpg">
    <p>图8-3：RAG系统能够针对用户的问题生成精准回答，并（在理想情况下）标注其参考的信息来源</p>
</div>


## 8.2 语言模型驱动的语义搜索实践

### 8.2.1 稠密检索

基于词嵌入的文本嵌入（向量化）技术可将语义信息映射至数值空间。如图8-4所示，这种空间映射使得语义相近的文本在向量空间中彼此邻近——例如文本1与文本2的语义相似度较高，而二者与文本3的语义距离相对较远。

<div align="center">
    <img src="./picture/8-4.jpg">
    <p>图8-4：词嵌入的几何化诠释：文本在向量空间中的位置分布反映其语义相关性</p>
</div>

基于此特性可构建智能搜索系统：当用户发起查询时，系统首先将查询语句编码至与文档库相同的向量空间，随后通过近邻搜索算法寻找空间距离最近的文档作为检索结果（图8-5）​。

<div align="center">
  <img src="./picture/8-5.jpg">
  <p>图8-5：稠密检索依赖于查询对象与相关结果在嵌入空间中的相似性</p>
</div>

从图8-5中的距离分布可以看出，对于该查询而言，文本2是最佳匹配结果，文本1次之。

这里可能引发两个值得探讨的问题。

· 是否应该将文本3纳入结果？这取决于系统设计者的决策。通常需要设置相似度阈值来过滤无关结果（特别是在语料库中缺乏相关文档时）​。

· 查询与最佳结果的语义是否真正相关？答案并非绝对肯定的。正因如此，语言模型需要通过问-答对训练来提升检索能力，这一过程将在第10章详细阐述。


图8-6展示了文档在嵌入前的分块处理流程。经过分块的文档通过嵌入模型转化为向量表示，最终存储在向量数据库中以备检索。

<div align="center">
  <img src="./picture/8-6.jpg">
  <p>图8-6：将外部知识库转换为向量数据库，通过嵌入处理实现知识库的智能化查询</p>
</div>

#### 稠密检索实例解析

以下以电影《星际穿越》(Interstellar)英文维基百科页面的检索为例，演示Cohere平台的稠密检索流程。具体实施步骤包括：

(1)对目标文本进行预处理和句子分割；

(2)生成句子的向量表示；

(3)建立搜索索引；

(4)执行搜索并分析结果。

首先导入必要的库

In [81]:
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
import os

# 换成你刚刚部署好的 Worker 地址
PROXY_URL = "https://cohere.zaxys916.workers.dev"


load_dotenv("../.env", override=True)   # 加 override=True
cohere_api_key = os.getenv("cohere_api_key")
# 关键：把 Cohere 的真实 API 地址作为 target 参数拼接
real_api_url = "https://api.cohere.com"

co = cohere.Client(
    api_key=cohere_api_key,
    base_url=f"{PROXY_URL}?target={real_api_url}"
) # api通过注册cohere获取，要有国外常用的邮箱


获取文本语料库并进行分块处理。我们以电影《星际穿越》的英文维基百科页面的开头部分的内容为例，首先获取原始文本数据，随后将其按句子进行切分

In [42]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subse quent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""
# 将文本分割成句子列表
texts = text.split('.')
# 清理空格和换行符
texts = [t.strip(' \n') for t in texts]

嵌入文本片段。现在我们开始对文本执行嵌入操作。我们将这些文本发送至Cohere API，即可获得每一段文本对应的向量表示。

In [ ]:
# 获取嵌入向量
response = co.embed(
    texts=texts,
    input_type="search_document",
    model="embed-multilingual-v3.0",   # 改这里
).embeddings

embeds = np.array(response)
print(embeds.shape) # 可能会有波动

(15, 1024)


输出结果为(15, 1024)，表明存在15个向量，每个向量的维度均为1024。


构建搜索索引。在实施搜索前，需预先构建搜索索引。该索引用于存储嵌入向量，其核心优化目标是实现海量数据点场景下的高效最近邻检索：

In [50]:
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)          # 输出 True
index.add(embeds.astype(np.float32))
print(index.ntotal)              # 应该输出 15

True
15


通过索引进行搜索。现在，我们可以使用任意查询语句来搜索数据集。只需将查询语句编码为嵌入向量，并将其输入索引系统，系统便会从维基百科页面中检索出语义最相近的句子。

接下来定义搜索函数：

In [48]:
def search(query, number_of_results=3):
    # 1. 获取查询的嵌入向量
    query_embed = co.embed(texts=[query],
                input_type="search_query",
                model="embed-multilingual-v3.0",).embeddings[0]
    # 2. 检索最近邻
    distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)
    # 3. 格式化结果
    texts_np = np.array(texts) # 将文本列表转换为numpy数组以便索引
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                                'distance': distances[0]})
    # 4. 打印并返回结果
    print(f"Query:'{query}'\nNearest neighbors:")
    return results

至此，我们已经可以编写查询指令并执行文本搜索了！

In [49]:
query = "how precise was the science"
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,0.909139
1,Caltech theoretical physicist and 2017 Nobel l...,1.220774
2,Cinematographer Hoyte van Hoytema shot it on 3...,1.306161


第一个结果与查询的相似度最高，因此成为最匹配的检索结果。从示例中可以看出，该结果完美解答了提出的问题。值得注意的是，这种结果在单纯使用关键词搜索时是不可能实现的，因为关键词搜索结果中排名最靠前的条目并未包含查询语句中的原始关键词。

为验证这一现象，我们可以定义一个关键词搜索函数。这里采用BM25算法，该算法是当前使用最广泛的词汇搜索方法之一。

In [54]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))
bm25 = BM25Okapi(tokenized_corpus)
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query) 
    ##### BM25搜索（词汇搜索） #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

100%|██████████| 15/15 [00:00<?, ?it/s]


现在，当我们针对同一查询进行搜索时，得到的结果与稠密检索的呈现方式有所不同：

In [55]:
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.793	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.377	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


注意，尽管第一个结果与查询都包含science这个词，但它并未真正回答问题。

#### 稠密检索的缺陷

理解稠密检索的局限性及其解决方案具有重要意义。例如，当文本中完全不存在答案时会发生什么？系统仍会返回结果及其相似度距离

在这种情况下，一种可行的方法是设定相关性阈值，例如设置最大距离阈值。而许多搜索系统会将能得到的最佳匹配结果呈现给用户，由用户自行判断相关性。通过追踪用户对搜索结果的点击行为及满意度反馈，可以持续优化搜索系统的迭代版本。

稠密检索的另一短板在于无法精准匹配特定短语，这类场景更适合关键词匹配技术。这正是建议采用混合搜索（结合语义搜索与关键词搜索）而非单纯依赖稠密检索的重要原因。

当将稠密检索系统应用于其训练数据之外的领域时，其性能也会显著下降。例如，若检索模型基当将稠密检索系统应用于其训练数据之外的领域时，其性能也会显著下降。例如，若检索模型基

需要特别指出的是，本示例中的每个句子都包含独立信息单元，而我们所展示的查询恰好精准对应这些信息单元。但答案分布于多个句子中的复杂问题应如何处理？这揭示了稠密检索系统的关键设计参数：如何实现长文本的最优分块处理？为何必须进行分块处理？

#### 长文本分块策略

Transformer语言模型的上下文长度限制带来了技术挑战——我们无法输入超过模型词元限制的超长文本。那么应如何有效嵌入长文本呢？

图8-7展示了两种主流方案：单文档单向量方案与单文档多向量方案。

<div align="center">
  <img src="./picture/8-7.jpg">
  <p>图8-7：虽然采用单个向量表示整个文档可行，但对篇幅较长的文档建议采用分块嵌入方案</p>
</div>

单文档单向量方案。该方案使用单个向量表示整个文档。常见实现方式包括以下两种。

· 仅嵌入文档的代表性段落，忽略剩余内容。例如仅嵌入标题或文档开头部分。这种方式虽适用于快速搭建演示系统，但会导致大量信息未被索引而无法检索。该方法适合文档首段即能概括核心观点的情况（如维基百科条目）​，但在实际系统中并非最佳选择，因其会排除大量可检索信息。

· 将文档分割为多个块并对各块进行嵌入处理，随后将这些块聚合为单个向量。常用聚合方式是对各向量取平均值，但此方式存在信息高度压缩的缺陷，导致文档中的大量细节丢失。

这种方案或许能满足某些信息检索需求，却难以覆盖其他情况。在实际应用中，用户往往需要搜索文章中的特定信息片段，若这些概念能拥有独立向量则更有利于精准捕获。

单文档多向量方案。该方案将文档切分为较小的块并进行块级嵌入，使搜索索引转变为块嵌入索引，而非文档整体嵌入索引。图8-8展示了多种文本分块方式的对比效果。

<div align="center">
  <img src="./picture/8-8.jpg">
  <p>图8-8：不同分块方法对输入文本的分割效果，其中重叠块能有效防止上下文割裂</p>
</div>

分块策略的优势在于实现文本的完整覆盖，使向量能够捕捉文本中的独立语义单元，从而构建表达力更强的搜索索引。图8-9列举了若干典型的实现方式。

<div align="center">
    <img src="./picture/8-9.jpg">
    <p>图8-9：文档分块生成嵌入向量的多种方式</p>
</div>

针对长文本的最佳分块方式，要根据系统需处理的文本类型和查询特征进行选择。

· 以句子作为独立块的处理方式可能粒度过小，导致向量无法捕捉足够的上下文信息。

· 以段落为单位进行分块是更优的选择。当文本段落较短时，这种方法效果良好；若段落较长，则建议每3～8个句子划分为一个块。

· 某些文本块的含义高度依赖上下文，可通过以下方式增强上下文相关性。

· 在块中附加文档标题。

· 引入一部分上下文内容。通过构建重叠块结构（即相邻块包含部分重复文本）​，可有效地保留上下文信息。图8-10所示即为这种方式的典型应用。

<div align="center">
    <img src="./picture/8-10.jpg">
    <p>图8-10：采用重叠式文本分块策略可有效保留不同片段间的上下文相关性</p>
</div>

随着稠密检索技术的持续演进，更多创新的分块策略正在涌现——部分方案已开始利用LLM实现动态智能分块，以生成语义连贯的文本单元

#### 最近邻搜索与向量数据库

完成查询嵌入后，如图8-11所示，我们需要在文档库中检索与之最相似的向量。最基础的实现方式是直接计算查询向量与文档库向量之间的距离。对于数千至数万量级的向量，使用NumPy即可高效完成这种计算。

<div align="center">
    <img src="./picture/8-11.jpg">
    <p>图8-11：如第4章所述，通过比较向量相似度可快速定位与查询最匹配的文档</p>
</div>


当处理百万量级的向量时，建议采用Annoy或FAISS等近似最近邻(approximate nearestneighbor，ANN)搜索库进行优化检索。这些工具可在毫秒级响应时间内处理海量数据，部分方案还可借助GPU加速和分布式集群部署实现超大规模索引的高效服务。

另一类解决方案是专为向量检索设计的数据库系统（如Weaviate、Pinecone）​。这类向量数据库支持动态增删向量而无须重建索引，并提供向量距离之外的过滤搜索、自定义搜索等高级功能。

#### 面向稠密检索的嵌入模型微调

在检索场景中，优化目标需要从词元嵌入扩展至文本级语义嵌入。该过程的核心在于构建由查询语句和相关文档组成的训练数据集。

以某数据集中的例句“Interstellar premiered on October26, 2014, in Los Angeles”​（​《星际穿越》于2014年10月26日在洛杉矶首映）为例，其对应的有效查询可能如下所示。

· 相关查询1：​“Interstellar release date”​（​《星际穿越》上映日期）​。

· 相关查询2：​“When did Interstellar premiere”​（​《星际穿越》是什么时候首映的）​。

微调过程的目标是使这些查询的嵌入向量更接近目标句子的嵌入向量。同时，模型需要处理与句子无关的查询，例如下面这个例子。

 不相关查询：​“Interstellar cast”​（​《星际穿越》演员阵容）​。

基于这些样本，我们得到三组数据——两对正例和一对负例。如图8-12所示，假设微调前这三个查询与结果文档的嵌入距离相等——这种情况也算合理，因为它们都涉及《星际穿越》​。

<div align="center">
    <img src="picture/8-12.jpg">
    <p>图8-12：在微调前，相关与不相关查询的嵌入向量可能均与特定文档邻近</p>
</div>

微调的核心作用是拉近相关查询与文档的距离，同时推离不相关查询。这种效果可通过图8-13直观呈现。
<div align="center">
    <img src="picture/8-13.jpg">
    <p>图8-13：经过微调后，文本嵌入模型利用提供的相关与不相关文档示例优化搜索任务表现，更贴合数据集的相关性定义</p>
</div>


### 8.2.2 重排序

许多组织已构建自有搜索系统。对于这些组织而言，将语言模型整合至搜索流程的最终环节是更便捷的实现方式。此环节通过调整搜索结果顺序提升查询的相关性，能显著改善搜索质量——微软必应正是采用类似BERT的模型实现了这一优化。图8-14展示了作为两阶段搜索系统中第二阶段的重排序的架构。

<div align="center">
    <img src="picture/8-14.jpg">
    <p>许多组织已构建自有搜索系统。对于这些组织而言，将语言模型整合至搜索流程的最终环节是更便捷的实现方式。此环节通过调整搜索结果顺序提升查询的相关性，能显著改善搜索质量——微软必应正是采用类似BERT的模型实现了这一优化。图8-14展示了作为两阶段搜索系统中第二阶段的重排序的架构。</p>
</div>

#### 重排序示例

重排器接收搜索查询与一组搜索结果，返回按相关性优化排序的文档列表，使相关性最高的结果位于前列。Cohere的Rerank端点提供了一种简单的方式来使用重排器，仅需传入查询和文本即可获得优化排序，无须训练或调参

In [57]:
query = "how precise was the science"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15232232),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subse quent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.05086082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [58]:
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)

0 0.15232232 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.05086082 The film had a worldwide gross over $677 million (and $773 million with subse quent re-releases), making it the tenth-highest grossing film of 2014
2 0.0350424 Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


这表明重排器对第一个结果更具信心，为其分配了超过0.16的相关性分数，而其他结果的相关性分数则显著偏低。

在这个基础示例中，我们向重排器传递了全部15个文档。但在实际应用中，索引可能包含成千上万个条目，通常需要先筛选出100或1000个候选结果，再将其提交给重排器。这个筛选过程被称为搜索流程的第一阶段检索。

第一阶段检索可采用关键词搜索、稠密检索，或是更优的方案——结合两者的混合搜索。通过回顾前面的示例，我们可以看到，在关键词搜索系统后加入重排器是如何提升系统性能的。

我们调整关键词搜索函数的工作流程：首先通过关键词搜索获取前10个结果，随后使用重排器从中精选出最优的3个结果

In [59]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)
    ##### BM25搜索（词汇搜索）#####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))
    # 添加重排序
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]
    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    # print(results.results)
    for hit in results.results:
        # print(hit)
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

现在，我们可以发送查询请求：首先检查关键词搜索的结果，从中筛选出前10个相关结果，再将其传递给重排器进行处理。

In [60]:
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.793	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.377	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.152	It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
	0.051	The film had a worldwide gross over $677 million (and $773 million with subse quent re-releases), making it the tenth-highest grossing film of 2014
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


通过观察可以发现，关键词搜索仅对均包含部分关键词的两个结果进行评分。在重排序后的第二组结果中，重排器成功将第二个结果提升为与查询相关性最高的结果。虽然这只是一个简单的示例，但足以让我们直观感受重排序的效果。在实际应用中，这样的处理流程能显著提升搜索质量。例如在多语言基准测试MIRACL中，重排器可使性能指标nDCG@10从36.5提升至62.8（本章后续将详细说明评估方法）​。

#### 使用sentence-transformers实现开源检索与重排序

若需在本地部署检索与重排序系统，可采用sentence-transformers(SBERT)库。具体配置方法请参考官方文档，并查阅“Retrieve & Re-Rank”​（检索与重排序）部分获取详细的步骤说明和代码实现。

#### 重排序模型工作机制

构建LLM搜索重排器的常规方法是将查询与每个候选结果共同输入交叉编码器架构的LLM。如图8-15所示，这种机制允许模型同时分析查询文本与文档内容后生成相关性评分。尽管文档采用批量处理方式，但每个文档都会单独跟查询进行匹配评估。最终模型根据这些分数重新排列搜索结果。该技术在论文“Multi-Stage Document Rankingwith BERT”中有详尽阐述，学界常称其为monoBERT方法。

<div align="center">
    <img src="picture/8-15.jpg">
    <p>图8-15：重排器通过联合分析文档与查询生成相关性分数</p>
</div>

这种基于相关性分数的搜索机制本质上可视为分类任务。模型接收输入后输出0和1之间的分数，0代表完全不相关，1代表高度相关。这与第4章讨论的分类问题原理相通。

欲深入了解LLM在搜索领域的发展脉络，强烈推荐阅读“Pretrained Transformers for TextRanking: BERT and Beyond”​，该论文系统梳理了截至2021年的相关技术演进

### 8.2.3 检索评估指标体系

语义搜索系统的评估沿用信息检索(informationretrieval，IR)领域的经典指标。我们重点解析其中最具代表性的指标之一：均值平均精确率[插图](mean average precision，mAP)。

完整的搜索系统评估框架包含三大要素：文档库、查询集合，以及表明查询与文档对应关系的相关性判断。如图8-16所示，这些组件共同构成评估基础。

<div align="center">
    <img src="picture/8-16.jpg">
    <p>图8-16：评估搜索系统所需的测试套件构成，其中包含查询集合以及表明查询与库中文档对应关系的相关性判断</p>
</div>

基于该测试套件，我们可进一步探讨搜索系统的评估方法。让我们从简单的案例入手：假设将查询1输入两个搜索系统后，我们获得两组结果。若将返回结果限定为三个（如图8-17所示）​，即可展开对比分析。

<div align="center">
    <img src="picture/8-17.jpg">
    <p>图8-17：使用同一个查询对两个搜索系统进行测试时，各自输出的前序结果对比</p>
</div>

要判定系统优劣，需参照该查询的相关性标注数据。图8-18直观呈现了两个系统的返回结果中相关文档的分布情况。

<div align="center">
    <img src="picture/8-18.jpg">
    <p>图8-18：通过对照测试套件的相关性标注，我们能够清晰判断系统1的检索效果优于系统2</p>
</div>

此案例中系统1的优越性显而易见。直觉上，我们可以直接统计各系统召回的相关结果数量——系统1在三个结果中命中两个相关文档，而系统2仅命中一个。但若如图8-19所示，两个系统在三个结果中都只召回一个相关文档，但排序位置不同，又该如何评判？

<div align="center">
    <img src="picture/8-19.jpg">
    <p>图8-19：需要设计能够区分排序优劣的评分机制——即使两个系统在前三个结果中均只包含一个相关文档，也应体现系统1将相关文档置于更高排位的优势</p>
</div>

此情境下，我们可直观认为系统1表现更优，因为其将相关结果放置在更重要的首位。但如何将这种优势转化为可量化的数值指标？

针对此类情况，通常采用平均精确率(averageprecision，AP)进行评分：系统1在该查询上的得分为1，而系统2的得分仅为0.3。接下来我们将解析平均精确率的计算逻辑，并说明如何通过该指标基于测试套件的全部查询综合评估系统表现。

#### 基于平均精确率的单查询评分

基于单个查询对搜索系统进行评分时，我们聚焦于相关文档的识别与排序。首先考察测试套件中仅含单个相关文档的查询场景。

第一种情况很简单：搜索系统将相关结果（这是该查询唯一可用的相关结果）置于首位。这使得系统获得了满分1。计算过程如图8-20所示，在第一个位置存在一个相关结果，因此位置1的精确率达1（计算方法为前k个位置的相关结果的数量除以当前查看的位置序号）​。

<div align="center">
    <img src="picture/8-20.jpg">
    <p>图8-20：计算平均精确率时需从位置1开始逐项计算各位置的精确率</p>
</div>

由于我们仅对相关文档进行评分，可以忽略不相关文档的分数并终止计算。但若系统将唯一相关结果置于第三位会如何？这种情况对评分的影响如图8-21所示，系统会因提前呈现不相关文档而受到得分惩罚。

<div align="center">
    <img src="picture/8-21.jpg">
    <p>图8-21：当系统将不相关文档排列在相关文档之前时，其精确率得分将受到惩罚</p>
</div>

接下来我们观察包含多个相关文档的查询案例。如图8-22所示，计算过程中需将所有相关文档处k个结果的精确率纳入平均值的计算。

<div align="center">
    <img src="picture/8-22.jpg">
    <p>图8-22：对于含多个相关文档的查询，平均精确率需综合所有相关文档处k个结果的精确率</p>
</div>


#### 基于均值平均精确率的多查询评分

在理解k个结果的精确率与单查询平均精确率后，我们可将其扩展至适用于测试套件中所有查询的评估指标——均值平均精确率。如图8-23所示，该指标通过取各查询平均精确率的均值计算得出。

<div align="center">
    <img src="picture/8-23.jpg">
    <p>图8-23：均值平均精确率考虑了系统在测试套件中每个查询的平均精确率得分，通过对这些得分取均值，生成一个单一指标，便于比较不同搜索系统的性能</p>
</div>

你可能会疑惑，为什么同样涉及“取平均数”的操作，要分别称“平均”和“均值”​。这应该是出于美观和通顺的考虑，​“均值平均精确率”要好于“平均平均精确率”​。

现在我们已经有了可用于系统间横向对比的单一指标。若需深入了解信息检索评估指标，可参阅Christopher D. Manning、PrabhakarRaghavan和Hinrich Schütze合著的Introduction to Information Retrieval（Cambridge University Press出版）中“Evaluation in Information Retrieval”一章。

除均值平均精确率外，搜索系统还常使用归一化折损累积增益(normalized discounted cumulativegain，nDCG)作为评估指标。该指标具有更精细的考量维度，因为在测试套件和评分机制中，文档的相关性并非二元的（只有相关与不相关）​，而是允许标注不同等级的相关程度。

## 8.3 RAG

随着LLM的大规模应用，用户开始频繁向其提问并期待事实性回应。模型虽然能正确回答部分问题，但也会出现大量看似自信实则错误的答案。业界主流解决方案是采用RAG技术，该技术最早在2020年的论文“Retrieval-AugmentedGeneration for Knowledge-Intensive NLPTasks”中提出，其架构如图8-24所示

<div align="center">
    <img src="picture/8-24.jpg">
    <p>图8-24：基础RAG流程包含检索与生成两个核心环节。LLM接收用户问题时，检索模块获取的信息将作为提示词被输入LLM，进而生成基于事实的答案</p>
</div>


RAG系统兼具检索与生成的双重能力，可视为传统生成系统的升级版：既有效减少了幻觉现象，又显著提升了回答的事实准确性。该技术还支持“与数据对话”(chat with my data)的应用场景，使企业和个人能够将LLM与内部数据或特定数据源（如书籍内容）对接。

这种模式同样适用于搜索引擎领域。当前越来越多的搜索引擎（例如Perplexity、Microsoft Bing AI和Google Gemini）正在集成LLM，用于生成搜索结果摘要或直接回答用户提问。

### 8.3.1 从搜索到RAG

现在我们尝试将普通搜索系统升级为RAG系统，核心方法是在搜索流程末端接入LLM。具体实现方式是将用户的问题与检索获得的前若干个相关文档共同输入LLM，使其基于检索提供的上下文生成答案。图8-25展示了该过程的典型示例。

<div align="center">
    <img src="picture/8-25.jpg">
    <p>图8-25：生成式搜索在搜索流程的末端生成答案和摘要，同时引用其来源（由搜索系统的前序步骤返回）</p>
</div>

这种生成过程被称为基于知识的生成，因为检索系统提供的相关信息为模型构建了特定上下文，使其能够在目标领域内进行定向生成。延续前文嵌入式搜索的案例，图8-26直观展示了如何在搜索流程后衔接基于知识的生成环节。

<div align="center">
    <img src="picture/8-26.jpg">
    <p>图8-26：通过比较嵌入向量之间的相似度，找到与输入提示词相关性最高的信息。在将提示词提供给LLM之前，将其添加到提示词中</p>
</div>


### 8.3.2 示例：使用LLM API进行基于知识的生成

接下来我们了解如何在搜索结果后添加基于知识的生成步骤，构建首个RAG系统。本示例将使用Cohere的托管LLM（基于本章前文所述的搜索系统）​，通过嵌入式搜索获取相关性最高的文档后，将这些文档与问题共同输入co.chat端点，从而生成基于知识的答案

In [61]:
query = "income generated"
# 1.检索
# 我们将使用嵌入式搜索，但理想情况下应该使用混合搜索
results = search(query)
# 2.基于知识的生成
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents=docs_dict
)
print(response.text)

Query:'income generated'
Nearest neighbors:
The film Interstellar generated a worldwide gross of over $677 million, and $773 million with subsequent re-releases.


我们对部分文本进行了高亮标记，因为模型识别出这些文本片段来源于我们输入的第一个文档：

<div align="center">
    <img src="./picture/image.png">
    <p>citations=[ChatCitation(start=21, end=36, text='worldwide gross', document_ids=['doc_0']), ChatCitation(start=40, end=57, text='over $677 million', document_ids=['doc_0']), ChatCitation(start=62, end=103, text='$773 million with subsequent re-releases.', document_ids=['doc_0'])]
documents=[{'id': 'doc_0', text': 'The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'}]</p>
</div>

### 8.3.3 示例：使用本地模型的RAG

现在，让我们尝试使用本地模型复现这一基础功能。尽管较小的本地模型在性能上可能不及大型托管模型，且无法实现文本片段引用功能，但演示这一流程仍具有重要参考价值。首先，我们需要下载一个量化模型。

#### 加载生成模型

In [69]:
from langchain_community.llms import LlamaCpp
# 注意确保模型路径在你的系统上是正确的！
llm = LlamaCpp(
    model_path="../models/Phi-3-mini-4k-instruct-q4.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

#### 加载嵌入模型

现在，我们加载一个用于生成嵌入向量的语言模型。在本示例中，我们将选用BAAI/bge-small-en-v1.5模型。

In [77]:
from langchain_community.embeddings import LlamaCppEmbeddings

embedding_model = LlamaCppEmbeddings(
    model_path=r"../models/bge-small-en-v1.5/bge-small-en-v1.5-q8_0.gguf",
    n_ctx=512,
)

llama_model_loader: loaded meta data with 24 key-value pairs and 197 tensors from ../models/bge-small-en-v1.5/bge-small-en-v1.5-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = bert
llama_model_loader: - kv   1:                               general.name str              = bge-small-en-v1.5
llama_model_loader: - kv   2:                           bert.block_count u32              = 12
llama_model_loader: - kv   3:                        bert.context_length u32              = 512
llama_model_loader: - kv   4:                      bert.embedding_length u32              = 384
llama_model_loader: - kv   5:                   bert.feed_forward_length u32              = 1536
llama_model_loader: - kv   6:                  bert.attention.head_count u32              = 12
llama_model_loader: - kv   7:          bert.attenti

现在我们可以通过嵌入模型完成向量数据库的初始化流程：

In [78]:
from langchain_community.vectorstores import FAISS
# 创建本地向量数据库
db = FAISS.from_texts(texts, embedding_model)

decode: cannot decode batches with this context (calling encode() instead)
llama_perf_context_print:        load time =       1.34 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   411 tokens (    0.00 ms per token, 411000000.00 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =     190.02 ms /   412 tokens
llama_perf_context_print:    graphs reused =          0


#### RAG提示词

提示词模板在RAG流程中具有关键性作用，这是我们将相关文档信息传递给LLM的核心。为此，我们将创建名为context的附加输入变量，该变量专门用于向LLM提供检索所得的文档内容：

In [ ]:
# ============ 导入必要的模块 ============
# PromptTemplate：用于创建可复用的提示词模板，支持占位符替换
from langchain_core.prompts import PromptTemplate
# RunnablePassthrough：把输入原样传递到下一个环节，不修改
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser：把 LLM 的输出对象解析成纯字符串
from langchain_core.output_parsers import StrOutputParser

# ============ 创建提示词模板 ============
# 用 Phi-3 的对话格式（<|user|> ... <|end|> <|assistant|>）构建模板
# {context} 会被检索到的相关文档替换，{question} 会被用户问题替换
prompt = PromptTemplate.from_template(
    """<|user|>
Relevant information:
{context}
Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
)

# ============ 用 LCEL 构建 RAG 链 ============
# LCEL（LangChain Expression Language）用管道符 | 把各环节串起来
rag = (
    # 第 1 步：接收用户输入，同时并行执行两个动作
    #   - "context"：调用检索器 db.as_retriever()，从向量库取出相关文本
    #   - "question"：用 RunnablePassthrough() 把用户原始问题原样保留
    #   输入是字符串（用户问题），输出是 {"context": [...], "question": "..."}
    {"context": db.as_retriever(), "question": RunnablePassthrough()}

    # 第 2 步：把上一步的字典填入提示词模板
    #   {context} 和 {question} 会被对应的值替换，生成完整提示词
    | prompt

    # 第 3 步：把完整提示词交给 LLM（Phi-3）生成回答
    #   输出是一个 AIMessage 对象，不是纯文本
    | llm

    # 第 4 步：把 AIMessage 对象解析成纯字符串
    | StrOutputParser()
)


decode: cannot decode batches with this context (calling encode() instead)
llama_perf_context_print:        load time =       1.34 ms
llama_perf_context_print: prompt eval time =       3.55 ms /     5 tokens (    0.71 ms per token,  1408.05 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =       5.13 ms /     6 tokens
llama_perf_context_print:    graphs reused =          0


现在，我们可以调用模型并提出问题：

In [85]:
# ============ 调用 ============
# rag.invoke() 接收用户问题（字符串），返回 LLM 生成的答案（字符串）
answer = rag.invoke('Income generated')
print(answer)

decode: cannot decode batches with this context (calling encode() instead)
llama_perf_context_print:        load time =       1.34 ms
llama_perf_context_print: prompt eval time =       4.16 ms /     4 tokens (    1.04 ms per token,   961.31 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =       5.66 ms /     5 tokens
llama_perf_context_print:    graphs reused =          0


 The provided information does not directly relate to income generated. However, based on the context of Interstellar and its production elements mentioned in the documents, it can be inferred that the movie's production involved notable contributors like Caltech theoretical physicist Kip Thorne, who acted as a scientific consultant, potentially impacting the film's success and revenue. But specific income details are not provided in these documents.


与之前一样，我们可以通过调整提示词来控制模型的生成效果（例如回答长度和语气等）​。

### 8.3.4 高级RAG技术

#### 查询改写

当RAG系统作为聊天机器人时，若用户提问冗长或需要关联对话上下文，基础RAG在信息检索环节可能表现欠佳。此时，使用LLM将原始查询转化为更利于检索的简洁形式是一种有效的策略。例如：

用户提问：​“我们明天有一篇关于动物的作文要交。我喜欢企鹅，可以写关于企鹅的。但我也可以写海豚。它们是动物吗？也许是吧。我们写海豚吧。比如，它们生活在哪里？​”

这个原始查询应被改写为：

查询：​“海豚生活在哪里”

此类改写可通过特定提示词或API实现。例如，Cohere的co.chat便内置了专用的查询改写模式。

#### 多查询RAG

此方法扩展查询改写能力，支持针对复杂问题生成多个关联查询。例如：

用户提问：​“比较NVIDIA 2020年与2023年的财报。​”

理想情况是找到同时包含两年数据的文档，但更有效的做法是生成两个独立查询：

查询1：​“NVIDIA 2020年财报”

查询2：​“NVIDIA 2023年财报

随后将两次检索的最佳结果输入模型进行事实性回答。进一步改进，可赋予改写器自主判断能力：需要执行检索，或直接生成可靠的答案。

####  多跳RAG

针对需要分步推理的复杂问题，系统需执行连续检索。例如：

用户提问：​“2023年排名最靠前的汽车制造商有哪几个？它们是否都生产电动汽车？​”

处理流程如下：

第1步，查询1：​“2023年排名最靠前的汽车制造商”

基于检索结果（如丰田、大众和现代）​，生成后续查询：

第2步，查询1：​“丰田汽车公司电动汽车”

第2步，查询2：​“大众汽车集团电动汽车”

第2步，查询3：​“现代汽车公司电动汽车”

#### 查询路由

该技术使模型具备多数据源定向检索能力。例如：

· 用户提出人力资源相关问题→检索公司知识库（如Notion）

· 用户提出客户数据相关问题→检索CRM系统（如Salesforce）

#### 智能体RAG

至此，你可能已经意识到，前述增强功能正逐步将愈加复杂的任务交给LLM。这种演进依赖于LLM对信息价值的评估能力，以及其整合多源数据的处理能力。这种新特性使得LLM愈发接近于能在现实世界执行任务的智能体。值得注意的是，数据源本身亦可抽象为工具。正如我们已经见到基于Notion的搜索功能，同理应也能实现向Notion发布内容的技术路径。

需特别说明的是，并非所有LLM都具备本节讨论的RAG功能。截至本书撰写时，仅有少数头部托管模型尝试支持此类特性。值得关注的是，Cohere推出的Command R+在此类任务中表现卓越，且其开放权重版本也可供使用。

### 8.3.5 RAG效果评估

RAG模型的评估体系仍处于快速发展阶段。推荐阅读论文“Evaluating Verifiability inGenerative Search Engines”(2023)，该研究通过人工评估对比了多种生成式搜索系统，其评估框架包含四个核心维度。

流畅性(fluency):生成文本的语言流畅度与逻辑连贯性。

感知效用(perceived utility):感知效用(perceived utility)

感知效用(perceived utility):感知效用(perceived utility)

感知效用(perceived utility):引用内容对相关论断的支持的有效性。

尽管人工评估仍是黄金标准，但学界正探索通过LLM-as-a-judge范式实现自动化评估，即使用高性能LLM对生成结果进行多维度评分。Ragas便是实现此类评估的开源工具库，它还包含以下两个评估指标。

忠实度(faithfulness):答案与所提供上下文的一致性程度。

 答案相关性(answer relevance):答案与提问主题的契合度。

## 8.4 小结

本章系统探讨了语言模型在搜索系统中的创新应用。

· 稠密检索：基于文本嵌入相似性的检索机制，通过将搜索查询向量化，匹配最相近的文档嵌入。

· 重排器：以monoBERT为代表的系统，通过评估查询与候选文档的相关性分数实现结果排序优化。

· RAG：在搜索流程末端部署生成式LLM，基于检索所得文档生成附带引证的回答。

我们还介绍了一种可行的搜索系统评估方法。均值平均精确率允许我们为搜索系统评分，从而基于一组测试查询及已知的相关性数据进行系统间的比较。值得注意的是，RAG系统的评估需要涵盖多个维度，包括忠实度、流畅性等指标，这些维度既可以通过人工评估，也可以借助LLM-as-a-judge进行量化分析。